# SFT Training with Qwen 2.5 7b

The Notebook requires GPU for execution. This can be run on Runpod or any other cloud GPU provider.



## Installation

In [3]:
!python -m pip install --upgrade pip -q
!pip install uv -qU
!uv pip install unsloth -qU --system
!uv pip install wandb -qU --system

In [ ]:
# AFTER INSTALLATION - RESTART THE KERNEL

In [1]:
import os
os.environ["HF_HOME"] = "/workspace"
os.environ["HF_HUB_CACHE"] = "/workspace/hub" # (recommended) override just the repo cache
print(os.environ["HF_HOME"])

/workspace


In [2]:
#!uv pip freeze > requirements-vllm-unsloth-ddp.txt --system

Hugging face setup

In [3]:
#from huggingface_hub import HfFolder, login
from huggingface_hub import login
login()
'''
# Check if a token is already saved
if HfFolder.get_token() is None:
    login()  # Will prompt only if not logged in
'''

'\n# Check if a token is already saved\nif HfFolder.get_token() is None:\n    login()  # Will prompt only if not logged in\n'

Force Hugging Face to store downloaded models/tokenizers in '/workspace' instead of the default cache location.

In [4]:
import os

os.environ["WANDB_PROJECT"] = "recipe-sft"
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"

## Fine-tuning

In [5]:
# # Base/Instruct Models
model_slug = "Qwen/Qwen2.5-7B-Instruct"
enable_thinking = False # set true if using thinking with Qwen 3 models.

test_run = False # to only run a limited set of dataset rows.
# -------
max_seq_length = 1024
dtype = None # unsloth will set this automatically
load_in_4bit = False # Use 4bit quantization to reduce memory usage. Can be False.
load_in_8bit = False
# -------

# # Dataset
## To use a synthetic dataset
ft_dataset_name = "nijumich/recipieNLG_V1"            

# Question / evaluation criteria / answer column names (adjust to your dataset)
q_column = "input"
c_column = "output"
# c_column = "evaluation_criteria" # swap to "answer" for a hacky way to use a ground truth as criteria (def flawed)
a_column = "output"
# a_column = "evaluation_criteria" # can just use evaluation_criteria if dataset has none.

different_eval_dataset = True


In [6]:
# Helper to clear cuda without restarting the kernel.
from unsloth import FastLanguageModel
import torch
import gc, inspect, sys

import warnings
warnings.filterwarnings( "ignore", message="Trainer.tokenizer is now deprecated. You should use Trainer.tokenizer instead.", category=UserWarning, )

def clear_old_model_refs():
    """
    Delete `teacher` `model` and `tokenizer` (if they exist) from the caller’s
    local *and* global scope, then garbage-collect and free GPU cache.
    """

    # ── figure out the caller’s frame ────────────────────────────
    frm = inspect.currentframe().f_back
    caller_locals  = frm.f_locals
    caller_globals = frm.f_globals

    for var in ("teacher", "model", "tokenizer"):
        if var in caller_locals:
            try:
                del caller_locals[var]
                if var in sys.modules:   # rarely needed
                    del sys.modules[var]
                print(f"deleted local  {var}")
            except Exception as e:
                print(f"could not delete local {var}: {e}")

        if var in caller_globals:
            try:
                del caller_globals[var]
                print(f"deleted global {var}")
            except Exception as e:
                print(f"could not delete global {var}: {e}")

    # ── Python & CUDA cleanup ───────────────────────────────────
    gc.collect()
    torch.cuda.empty_cache()
    print("GPU cache cleared.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [7]:
clear_old_model_refs()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_slug,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    load_in_8bit = load_in_8bit,
    use_gradient_checkpointing="unsloth",
    # fast_inference=False # ADD THIS IN TO USE VLLM FOR AUTOREGRESSIVE FORWARD PASSES
    # cache_dir = "./" # not necessary if you have already downloaded the model with vllm as HF_HOME is set
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf, if you're not already logged in to hf
)

print(tokenizer.padding_side)

GPU cache cleared.
==((====))==  Unsloth 2026.4.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.252 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

unsloth/Qwen2.5-7B-Instruct does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
left


In [8]:
print(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584, padding_idx=151665)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)

In [9]:
rank=16 # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
lora_alpha=32 # 2 time of r. This is suggetsed to be stable heuristic

model = FastLanguageModel.get_peft_model(
    model,
    r = rank,
    
    # OR via specific module names
    target_modules = [
            "q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj", # don't train these if it's a MoE (not tested with unsloth, but should work for qwen3)
        ],
    # OR all linear layers (also not recommended)
    # target_modules = ["all-linear"], # to train all linear layers

    # modules_to_save = ["lm_head","embed_tokens"], # to full train embeddings
    lora_alpha = lora_alpha,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    # full_finetuning = False,
    random_state = 3407,
    use_rslora = True,  # rank stabilized LoRA
)

Unsloth 2026.4.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [10]:
# print(model)

In [10]:
model.print_trainable_parameters()

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


In [11]:
from datasets import load_dataset

offline = True
if offline: 
    dataset = load_dataset("json", data_files={"train": "/workspace/ADVANCED-fine-tuning/fine-tune/datasets/train_gold_80k.jsonl",
    "validation": "/workspace/ADVANCED-fine-tuning/fine-tune/datasets/val_3k.jsonl"})
    
    train_ds = dataset["train"]
    eval_ds = dataset["validation"]
    #test_ds = dataset["test"]
    ft_train_data = train_ds#train_ds.select(range(100))  # Take only first 100 rows
    ft_eval_data = eval_ds#eval_ds.select(range(100))  # Take only first 100 rows
    #ft_test_data = test_ds.select(range(100))  # Take only first 100 rows

else :
    ft_data = load_dataset(ft_dataset_name)
    ft_train_data = ft_data["train"]

    if different_eval_dataset is not None:
        ft_eval_data = different_eval_dataset
    else:
        if "validation" in ft_data:
            ft_eval_data = ft_data["validation"]
        else:
            ft_eval_data = None
            print("No eval data")

In [12]:
print(ft_train_data)

Dataset({
    features: ['instruction', 'input', 'output', 'recipe_id', 'title_normalized', 'ingredients_normalized', 'ingredients_bullets', 'ingredients_names_only', 'input_tokens', 'output_tokens', 'directions_normalized', 'ner_ingredients', 'input_tok_len_gold', 'output_tok_len_gold', 'token_ratio_gold', 'total_tok_len_gold'],
    num_rows: 80000
})


In [13]:
print(ft_train_data['input'][0])

Title: Mom'S Meatloaf

Ingredients:
- Eggs - 2
- Milk - 3/4 cup
- Onion - 1/2 cup
- Breadcrumbs - 2/3 cup
- Salt - 1 tsp
- Pepper - 1/8 tsp
- Rubbed sage - 1/2 tsp
- Ground beef - 1 1/2 lb
- Ketchup - 1 cup
- Brown sugar - 1/2 cup
- Worcestershire sauce - 1 tsp


In [14]:
# To down-select data.
if test_run: ft_train_data = ft_train_data.select(range(1000))
if test_run: ft_eval_data = ft_eval_data.select(range(100))

In [14]:
def formatting_func(batch):
    """Convert a *batch* of rows to a list[str] of chat-formatted prompts."""
    out = []
    
    # batch[q_column] is a list; iterate over it
    for title_and_ingredients,directions in zip(batch[q_column], batch[a_column]):

        messages = [
            {"role": "system", 
             "content": "You are a culinary assistant. "
                    "Write step-by-step cooking directions using the given title and ingredients. "
                    "Use all relevant ingredients"
                    "Do NOT repeat the ingredient list. "
                    "Use complete sentences."
                    "Use numbered steps with action verbs."},
            {"role": "user", "content": title_and_ingredients},
            {"role": "assistant", "content": directions}
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=enable_thinking,
        )

        # Ensures two bos tokens aren't added, because the later tokenisation adds one!
        bos = tokenizer.bos_token or "<bos>"
        if text.startswith(bos):
            text = text[len(bos):]

        out.append(text)

    return out        # ← length == batch size

In [15]:
print(formatting_func(ft_eval_data)[2])

<|im_start|>system
You are a culinary assistant. Write step-by-step cooking directions using the given title and ingredients. Use all relevant ingredientsDo NOT repeat the ingredient list. Use complete sentences.Use numbered steps with action verbs.<|im_end|>
<|im_start|>user
Title: Gumbo Succotash Recipe

Ingredients:
- Crab meat - 1 pkg
- Cooked shrimp - 1 pkg
- Warm sausage - 3 to 4
- Stewed tomatoes - 1 can
- Tomato sauce - 1 can
- Med. whole onion - 1
- Green pepper
- Salt - 1/4 tsp
- Pepper
- Sugar - 2 tsp
- Okra - cut-up
- Gumbo file<|im_end|>
<|im_start|>assistant
1. Brown okra with small amount of butter or possibly veg.
2. oil in skillet.
3. Add in warm sausages (cut-up).
4. Add in green pepper (cut-up).
5. Add in onion (cut-up).
6. Add in crab meat and shrimp.
7. Add in stewed tomatoes.
8. Add in tomato sauce and 2 cans water.
9. Simmer.
10. Add in gumbo file, salt, sugar, and pepper.
11. Serve over cooked rice.
12. Serves 6 to 8 people.<|im_end|>



In [16]:
from trl import SFTTrainer, SFTConfig
import wandb
from datetime import datetime

if wandb.run is not None:
    wandb.finish()

wandb.login()

per_device_train_batch_size = 64 #4 # reduce if you run out of VRAM.
gradient_accumulation_steps = 1 #int(32 / per_device_train_batch_size)
epochs = 1
learning_rate = 2e-5 #1e-4  # for a 1B model go for 2e-4, for 8B go for 2e-5, for 3B go for 1e-4, 30B go for 5e-6

# Get current timestamp
current_timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')

if 'ft_dataset_name' in globals() or 'ft_dataset_name' in locals():
    # The variable is defined, proceed with your logic
    if ft_dataset_name is not None:
        run_name = f"{model_slug.split('/')[-1]}-{ft_dataset_name.split('/')[-1][:15]}-{epochs}ep-{current_timestamp}"
    else:
        print("ft_dataset_name is not defined and a run name cannot be set to include it. Check and rerun this cell")
else:
    run_name = f"{model_slug.split('/')[-1]}-{dataset_name.split('/')[-1][:15]}-{epochs}ep-{current_timestamp}"

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: niju to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [17]:
# ──────────────────
# Decide if we evaluate
# ──────────────────
do_eval = ft_eval_data is not None          # True ⇢ we have a dataset
if do_eval:
    print(f"Will run evaluation using validation dataset")
else:
    print(f"Will not run evaluation, as no dataset was passed")

# ──────────────────
# Build training_args
# ──────────────────
from unsloth import is_bfloat16_supported

training_args = SFTConfig(
    per_device_train_batch_size = per_device_train_batch_size,
    per_device_eval_batch_size  = 2 ,#per_device_train_batch_size,
    gradient_accumulation_steps = gradient_accumulation_steps,
    num_train_epochs            = epochs,
    #max_steps=30,   # uncomment for shorter run
    
    logging_strategy = "steps",
    logging_dir      = f"logs/{model_slug.split('/')[-1]}",
    eval_strategy    = "steps",
    logging_steps    = 10, #1       # cannot be fractional (0.05 invalid)
    eval_steps       = 250, #0.02,      # ⚠️ must be int steps, not a fraction
    save_strategy    = "steps",
    save_steps       = 250,
    save_total_limit = 2,
    load_best_model_at_end = do_eval,
    metric_for_best_model = "eval_loss" if do_eval else None,
    greater_is_better = False if do_eval else None,
    
    bf16  = is_bfloat16_supported(),
    fp16  = not is_bfloat16_supported(),
    report_to = "wandb",
    seed      = 3407,
    output_dir = "outputs",

    gradient_checkpointing = True,
    gradient_checkpointing_kwargs = {"use_reentrant": True},
    remove_unused_columns = True,
    lr_scheduler_type     = "cosine",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Will run evaluation using validation dataset


#### Fine-tuning

In [18]:
num_gpus = torch.cuda.device_count()
gpu_tag = f"{num_gpus}gpu" if num_gpus > 0 else "cpu"

run_name = f"{run_name}-ft-{gpu_tag}"
print(f"Setting up for run: {run_name}")

training_args.run_name = run_name
training_args.logging_dir = f"./logs/{run_name}"

wandb.init(
    project=os.environ["WANDB_PROJECT"],
    name=run_name,
    group=model_slug.split("/")[-1],
    config={
        "model_slug": model_slug,
        "lora_rank": rank,
        "lora_alpha": lora_alpha,
        "use_rslora": True,
        "max_seq_length": max_seq_length,
        "per_device_train_batch_size": per_device_train_batch_size,
        "gradient_accumulation_steps": gradient_accumulation_steps,
        "epochs": epochs,
        "learning_rate": learning_rate,
        "lr_scheduler_type": training_args.lr_scheduler_type,
        "logging_steps": training_args.logging_steps,
        "eval_steps": training_args.eval_steps,
        "save_steps": training_args.save_steps,
        "load_in_4bit": load_in_4bit,
        "load_in_8bit": load_in_8bit,
        "num_gpus": num_gpus,
        "gpu_name": torch.cuda.get_device_name(0) if num_gpus > 0 else "cpu",
        "dataset_train_size": len(ft_train_data),
        "dataset_eval_size": len(ft_eval_data) if ft_eval_data is not None else 0,
    },
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ft_train_data,
    eval_dataset=ft_eval_data,
    args=training_args,
    formatting_func=formatting_func
)

Setting up for run: Qwen2.5-7B-Instruct-recipieNLG_V1-1ep-20260406-082755-ft-1gpu


wandb: Detected [huggingface_hub.inference] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/80000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/3000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [19]:
print(trainer.train_dataset)

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 80000
})


In [20]:
from unsloth.chat_templates import train_on_responses_only 

TEMPLATES = {
    "llama": (
        "<|start_header_id|>user<|end_header_id|>\n\n",
        "<|start_header_id|>assistant<|end_header_id|>\n\n",
    ),
    "gemma": (
        "<start_of_turn>user\n",
        "<start_of_turn>model\n",
    ),
    "qwen": (
        "<|im_start|>user\n",
        "<|im_start|>assistant\n", # No thinking
    ),
    "mistral": (
        "[INST]",
        "[/INST]",
    )
}

instruction_tag, response_tag = TEMPLATES["qwen"]   # ← change if needed

# masks everything between the instruction_part and response_part
trainer = train_on_responses_only(
    trainer,
    instruction_part = instruction_tag,
    response_part = response_tag,
    # force_match=False # comment out to set true for a cleaner masking
)

Map (num_proc=64):   0%|          | 0/80000 [00:00<?, ? examples/s]

Filter (num_proc=64):   0%|          | 0/80000 [00:00<?, ? examples/s]

Map (num_proc=64):   0%|          | 0/3000 [00:00<?, ? examples/s]

Filter (num_proc=64):   0%|          | 0/3000 [00:00<?, ? examples/s]

In [21]:
print(trainer.train_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 80000
})


In [22]:
tokenizer.decode(trainer.train_dataset[0]["input_ids"])

"<|im_start|>system\nYou are a culinary assistant. Write step-by-step cooking directions using the given title and ingredients. Use all relevant ingredientsDo NOT repeat the ingredient list. Use complete sentences.Use numbered steps with action verbs.<|im_end|>\n<|im_start|>user\nTitle: Mom'S Meatloaf\n\nIngredients:\n- Eggs - 2\n- Milk - 3/4 cup\n- Onion - 1/2 cup\n- Breadcrumbs - 2/3 cup\n- Salt - 1 tsp\n- Pepper - 1/8 tsp\n- Rubbed sage - 1/2 tsp\n- Ground beef - 1 1/2 lb\n- Ketchup - 1 cup\n- Brown sugar - 1/2 cup\n- Worcestershire sauce - 1 tsp<|im_end|>\n<|im_start|>assistant\n1. Beat eggs in a large bowl.\n2. Add the milk, onion, breadcrumbs, salt, pepper and sage. Add the beef and mix well. Shape into an oval loaf and place in a roasting pan.\n3. In a measuring cup, mix remaining ingredients and pour about 3/4 cup over the meatloaf. Spread evenly.\n4. Bake at 350 for 1 hour and 5 to 10 minutes. ( No pink should remain in the meatloaf).\n5. Drain and let stand 5 minutes before s

In [23]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[0]["labels"]]).replace(tokenizer.pad_token, " ")

'                                                                                                                                                                   1. Beat eggs in a large bowl.\n2. Add the milk, onion, breadcrumbs, salt, pepper and sage. Add the beef and mix well. Shape into an oval loaf and place in a roasting pan.\n3. In a measuring cup, mix remaining ingredients and pour about 3/4 cup over the meatloaf. Spread evenly.\n4. Bake at 350 for 1 hour and 5 to 10 minutes. ( No pink should remain in the meatloaf).\n5. Drain and let stand 5 minutes before slicing. Serve with remaining sauce mixture.<|im_end|>\n'

Check if Masking is working

In [24]:
# Test one example from your processed dataset
sample = trainer.train_dataset[0]
decoded_labels = tokenizer.decode([l for l in sample['labels'] if l != -100])
decoded_input = tokenizer.decode(sample['input_ids'])

print("--- FULL INPUT ---")
print(decoded_input)
print("\n--- WHAT THE MODEL ACTUALLY LEARNS (LABELS) ---")
print(decoded_labels)

# VERIFY: If decoded_labels contains ingredients, your assistant_start_idx is wrong.

--- FULL INPUT ---
<|im_start|>system
You are a culinary assistant. Write step-by-step cooking directions using the given title and ingredients. Use all relevant ingredientsDo NOT repeat the ingredient list. Use complete sentences.Use numbered steps with action verbs.<|im_end|>
<|im_start|>user
Title: Mom'S Meatloaf

Ingredients:
- Eggs - 2
- Milk - 3/4 cup
- Onion - 1/2 cup
- Breadcrumbs - 2/3 cup
- Salt - 1 tsp
- Pepper - 1/8 tsp
- Rubbed sage - 1/2 tsp
- Ground beef - 1 1/2 lb
- Ketchup - 1 cup
- Brown sugar - 1/2 cup
- Worcestershire sauce - 1 tsp<|im_end|>
<|im_start|>assistant
1. Beat eggs in a large bowl.
2. Add the milk, onion, breadcrumbs, salt, pepper and sage. Add the beef and mix well. Shape into an oval loaf and place in a roasting pan.
3. In a measuring cup, mix remaining ingredients and pour about 3/4 cup over the meatloaf. Spread evenly.
4. Bake at 350 for 1 hour and 5 to 10 minutes. ( No pink should remain in the meatloaf).
5. Drain and let stand 5 minutes before slici

#### Start training

In [25]:
# Check memory
#@title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A100-SXM4-80GB. Max memory = 79.252 GB.
14.416 GB of memory reserved.


In [26]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 80,000 | Num Epochs = 1 | Total steps = 1,250
O^O/ \_/ \    Batch size per device = 64 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (64 x 1 x 1) = 64
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
250,1.282662,1.258517
500,1.272065,1.238370
750,1.231959,1.225975
1000,1.259292,1.218304
1250,1.267101,1.216910


/usr/local/lib/python3.11/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.11/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.11/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

In [27]:
# Final memory results
#@title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")
wandb.finish()

18405.753 seconds used for training.
306.76 minutes used for training.
Peak reserved memory = 55.043 GB.
Peak reserved memory for training = 40.627 GB.
Peak reserved memory % of max memory = 69.453 %.
Peak reserved memory for training % of max memory = 51.263 %.


eval/loss,█▅▃▁▁
eval/runtime,▁▆█▅▆
eval/samples_per_second,█▃▁▄▃
eval/steps_per_second,█▃▁▄▃
train/epoch,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇█████
train/global_step,▁▁▁▂▂▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇██
train/grad_norm,█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁▁▂▁▁
train/learning_rate,▂▄▆███████▇▇▇▆▆▆▆▆▅▅▄▄▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁
train/loss,█▆▃▂▂▂▂▂▂▂▁▂▂▂▂▂▁▂▂▁▂▂▁▂▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁
eval/loss,1.21691
eval/runtime,135.1105


### Save and/or Push the Model to Hub

In [28]:
print(run_name)

Qwen2.5-7B-Instruct-recipieNLG_V1-1ep-20260406-082755-ft-1gpu


In [ ]:
# # # Manually shorten the run name and just re-run the push, if needed.
# run_name = "ddp-demo-{num-gpus}"

In [ ]:
# # Merge to 16bit (RECOMMENDED, should merge to a dequantized base model for best accuracy)
org = "nijumich"
# print(f"Saving and pushing as {run_name} and {org}/{run_name}")

# Just SAVE locally
#if True: model.save_pretrained_merged(f"{run_name}", tokenizer, save_method = "merged_16bit",)

# # # Save locally AND push to hub
if True: model.push_to_hub_merged(f"{org}/{run_name}", 
                                  tokenizer, 
                                  save_method = "merged_16bit",
                                   token = "hf_xx")

# Not a best option to expose the hf token here 

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Found HuggingFace hub cache directory: /workspace/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...


...


Unsloth: Merging weights into 16bit:  75%|███████▌  | 3/4 [06:25<02:04, 124.94s/it]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [06:53<00:00, 103.27s/it]


Unsloth: Merge process complete. Saved to `/workspace/ADVANCED-fine-tuning/fine-tune/nijumich/Qwen2.5-7B-Instruct-recipieNLG_V1-1ep-20260406-082755-ft-1gpu`


In [30]:
# Print the run name
print(run_name)

Qwen2.5-7B-Instruct-recipieNLG_V1-1ep-20260406-082755-ft-1gpu


Notes: 

    This notebook is based on a Notebook by [Trelis Research](https://trelis.com/about).
    Available at [Trelis.com/ADVANCED-fine-tuning]().